# Module 08 — Notebook 01: Pixel Classification Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_08_RL_For_Image_Segmentation/01_Pixel_Classification_Agent/pixel_classification_agent.ipynb)

---

## Learning Objectives

1. Formulate pixel-level image segmentation as a Markov Decision Process (MDP).
2. Design a state representation based on local pixel patches.
3. Implement a Q-learning agent that assigns each pixel to a semantic class.
4. Train the agent on synthetic images with clear geometric regions.
5. Evaluate segmentation quality using Intersection over Union (IoU) and visualize progress.

In [ ]:
!pip install torch torchvision numpy matplotlib scikit-image

In [ ]:
# Download REAL open-source dataset for segmentation (TINY - under 200MB total)
import torchvision
import numpy as np

# CIFAR-10: 60,000 real photos (already cached if downloaded before)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
real_images = [np.array(cifar10[i][0]) for i in range(500)]
print(f"✅ CIFAR-10: {len(real_images)} real photos loaded (32x32 RGB)")

# FashionMNIST: 60,000 real clothing images (30MB only!)
fashion = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)
print(f"✅ FashionMNIST: {len(fashion)} real clothing images (28x28)")
print(f"   Classes: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Boot")

# Generate segmentation masks from CIFAR-10 using simple thresholding
# (This gives us real images with automatic pseudo-masks - no large dataset needed!)
def generate_pseudo_masks(images, threshold=128):
    """Generate binary masks from real images using Otsu-like thresholding."""
    masks = []
    for img in images:
        gray = np.mean(img, axis=2)
        mask = (gray > np.median(gray)).astype(np.uint8)
        masks.append(mask)
    return masks

pseudo_masks = generate_pseudo_masks(real_images)
print(f"✅ Generated {len(pseudo_masks)} segmentation masks from real images")
print(f"   Total download: ~200MB (CIFAR-10 + FashionMNIST)")

## Deep Derivation: Segmentation Metrics and MDP Theory

### Step 1: IoU (Intersection over Union) — Full Derivation
For class $k$:
$$\text{IoU}_k = \frac{|P_k \cap G_k|}{|P_k \cup G_k|}$$

**Expand using inclusion-exclusion:**
$$|P_k \cup G_k| = |P_k| + |G_k| - |P_k \cap G_k|$$

Therefore:
$$\text{IoU}_k = \frac{\text{TP}_k}{\text{TP}_k + \text{FP}_k + \text{FN}_k}$$

**Proof:** 
- $|P_k \cap G_k| = $ True Positives (predicted $k$ AND truly $k$)
- $|P_k| - |P_k \cap G_k| = $ False Positives (predicted $k$ but NOT truly $k$)
- $|G_k| - |P_k \cap G_k| = $ False Negatives (truly $k$ but NOT predicted $k$)
- Union $= \text{TP} + \text{FP} + \text{FN}$ $\blacksquare$

### Step 2: Dice Coefficient and its Relationship to IoU
$$\text{Dice}_k = \frac{2|P_k \cap G_k|}{|P_k| + |G_k|} = \frac{2\text{TP}}{2\text{TP} + \text{FP} + \text{FN}}$$

**Relationship to IoU:**
$$\text{Dice} = \frac{2 \cdot \text{IoU}}{1 + \text{IoU}}, \quad \text{IoU} = \frac{\text{Dice}}{2 - \text{Dice}}$$

**Proof:**
$$\frac{2 \cdot \text{IoU}}{1 + \text{IoU}} = \frac{2\frac{\text{TP}}{\text{TP+FP+FN}}}{1 + \frac{\text{TP}}{\text{TP+FP+FN}}} = \frac{2\text{TP}}{2\text{TP+FP+FN}} = \text{Dice} \quad \blacksquare$$

### Step 3: Cross-Entropy Loss per Pixel
For pixel $(i,j)$ with true class $y_{ij}$ and predicted probabilities $\hat{p}_{ij,k}$:
$$L_{CE} = -\frac{1}{HW}\sum_{i,j}\sum_{k=0}^{K-1} \mathbb{1}[y_{ij}=k] \log \hat{p}_{ij,k}$$

### Step 4: MDP Reward Shaping for Segmentation
**Potential-based reward shaping** (guarantees same optimal policy):
$$r'(s,a,s') = r(s,a,s') + \gamma\Phi(s') - \Phi(s)$$

**Theorem (Ng et al. 1999):** The optimal policy is invariant to potential-based shaping.

**For segmentation:** $\Phi(s) = $ current mIoU estimate

### Step 5: State Space Complexity Analysis
Patch size $N \times N$ with $C$ channels, quantized to $B$ bins:
$$|\mathcal{S}| = B^{N^2 \cdot C}$$

For $N=5$, $C=3$, $B=8$: $|\mathcal{S}| = 8^{75} \approx 10^{67}$ — needs function approximation!

### Step 6: Convergence Rate of Pixel Q-Learning
For tabular Q-learning with pixel classification:
$$\|Q_T - Q^*\|_\infty \leq (1-\alpha\gamma)^T \|Q_0 - Q^*\|_\infty$$

After $T = \frac{1}{\alpha(1-\gamma)}\log\frac{1}{\epsilon}$ steps: error $\leq \epsilon$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import torch.nn as nn
import torch.optim as optim
from collections import defaultdict, deque
import random
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

print('All imports successful.')

---
## 1. Mathematical Formulation: MDP for Pixel Classification

We cast pixel-level segmentation as a **Markov Decision Process** $\mathcal{M} = (\mathcal{S}, \mathcal{A}, R, T, \gamma)$.

### State Space $\mathcal{S}$

The agent visits each pixel $(i,j)$ in raster-scan order. The state at pixel $(i,j)$ is the **local $N \times N$ patch** centred on it:

$$
s_{i,j} = \text{Patch}(I, i, j, N) = \{I(i+\Delta r, j+\Delta c) \mid -\lfloor N/2\rfloor \le \Delta r, \Delta c \le \lfloor N/2\rfloor\}
$$

Each patch is flattened into a feature vector $\mathbf{x} \in \mathbb{R}^{N^2 C}$ where $C$ is the number of image channels.

### Action Space $\mathcal{A}$

The agent selects a class label for the current pixel:

$$
\mathcal{A} = \{0, 1, \ldots, K-1\}
$$

where $K$ is the number of segmentation classes.

### Reward Function $R$

We define a **per-pixel reward** based on correctness:

$$
r(s_{i,j}, a) = \begin{cases} +1 & \text{if } a = y_{i,j} \\ -1 & \text{if } a \neq y_{i,j} \end{cases}
$$

Additionally, after processing all pixels we compute the **IoU bonus**:

$$
\text{IoU}_k = \frac{|\hat{Y}_k \cap Y_k|}{|\hat{Y}_k \cup Y_k|}, \qquad \text{mIoU} = \frac{1}{K}\sum_{k=0}^{K-1} \text{IoU}_k
$$

### Transition Function $T$

The transition is **deterministic**: after classifying pixel $(i,j)$ the agent moves to the next pixel $(i, j+1)$ (or $(i+1, 0)$ at row end).

### Q-Learning Update

$$
Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \Big]
$$

---
## 2. Synthetic Dataset Generation

We create a synthetic grayscale image with **three classes**: background, circle, and square.

In [ ]:
def create_synthetic_image(size=64):
    """Generate a synthetic image with background (0), circle (1), and square (2)."""
    img = np.zeros((size, size), dtype=np.float32)
    gt = np.zeros((size, size), dtype=np.int64)

    # Circle
    cy, cx, radius = size // 3, size // 3, size // 6
    yy, xx = np.ogrid[:size, :size]
    circle_mask = (yy - cy)**2 + (xx - cx)**2 <= radius**2
    img[circle_mask] = 0.7
    gt[circle_mask] = 1

    # Square
    sq_y, sq_x, sq_half = 2 * size // 3, 2 * size // 3, size // 8
    img[sq_y - sq_half:sq_y + sq_half, sq_x - sq_half:sq_x + sq_half] = 0.4
    gt[sq_y - sq_half:sq_y + sq_half, sq_x - sq_half:sq_x + sq_half] = 2

    # Add slight noise
    img += np.random.normal(0, 0.05, img.shape).astype(np.float32)
    img = np.clip(img, 0, 1)
    return img, gt

IMG_SIZE = 32
image, ground_truth = create_synthetic_image(IMG_SIZE)

cmap_gt = mcolors.ListedColormap(['black', 'dodgerblue', 'orange'])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Synthetic Image'); axes[0].axis('off')
axes[1].imshow(ground_truth, cmap=cmap_gt, vmin=0, vmax=2); axes[1].set_title('Ground Truth (3 classes)'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## Deep Derivation: Differentiable IoU and Gradient-Based Optimization

### Step 1: Soft IoU for Backpropagation

Hard IoU is non-differentiable (involves set cardinality). We define a **soft IoU** using predicted probabilities $\hat{p}_{ij,k} \in [0,1]$:

$$\text{Soft-IoU}_k = \frac{\sum_{i,j} \hat{p}_{ij,k} \cdot g_{ij,k}}{\sum_{i,j} \hat{p}_{ij,k} + \sum_{i,j} g_{ij,k} - \sum_{i,j} \hat{p}_{ij,k} \cdot g_{ij,k}}$$

where $g_{ij,k} = \mathbb{1}[y_{ij} = k]$ is the one-hot ground truth.

### Step 2: Gradient of Soft IoU w.r.t. Predictions

Let $A = \sum_{i,j} \hat{p}_{ij,k} \cdot g_{ij,k}$ (soft intersection) and $B = \sum_{i,j} \hat{p}_{ij,k} + \sum_{i,j} g_{ij,k} - A$ (soft union).

$$\text{Soft-IoU}_k = \frac{A}{B}$$

By the quotient rule:

$$\frac{\partial \text{Soft-IoU}_k}{\partial \hat{p}_{ij,k}} = \frac{\frac{\partial A}{\partial \hat{p}_{ij,k}} \cdot B - A \cdot \frac{\partial B}{\partial \hat{p}_{ij,k}}}{B^2}$$

**Computing partial derivatives:**

$$\frac{\partial A}{\partial \hat{p}_{ij,k}} = g_{ij,k}, \qquad \frac{\partial B}{\partial \hat{p}_{ij,k}} = 1 - g_{ij,k}$$

**Therefore:**

$$\frac{\partial \text{Soft-IoU}_k}{\partial \hat{p}_{ij,k}} = \frac{g_{ij,k} \cdot B - A \cdot (1 - g_{ij,k})}{B^2} = \frac{g_{ij,k} \cdot (B + A) - A}{B^2}$$

### Step 3: IoU Loss and its Properties

The IoU loss is $\mathcal{L}_{\text{IoU}} = 1 - \text{mSoft-IoU}$, where:

$$\text{mSoft-IoU} = \frac{1}{K}\sum_{k=0}^{K-1} \text{Soft-IoU}_k$$

**Property 1 (Bounded):** $\mathcal{L}_{\text{IoU}} \in [0, 1]$ since $\text{Soft-IoU}_k \in [0,1]$.

**Property 2 (Class-balanced):** Unlike cross-entropy, IoU loss inherently balances classes because each class contributes equally through the mean, regardless of class size.

**Proof:** For a rare class $k$ with $|G_k| \ll HW$, cross-entropy's contribution is $\sim |G_k|/HW$ (negligible), but $\text{IoU}_k$ ranges in $[0,1]$ regardless of class size. $\blacksquare$

### Step 4: Connection to Dice Loss

The Dice loss $\mathcal{L}_{\text{Dice}} = 1 - \frac{2\sum_i p_i g_i}{\sum_i p_i + \sum_i g_i}$ is monotonically related to IoU loss:

$$\text{Dice} = \frac{2\text{IoU}}{1 + \text{IoU}} \implies \mathcal{L}_{\text{Dice}} \leq \mathcal{L}_{\text{IoU}} \leq 2\mathcal{L}_{\text{Dice}}$$

**Proof of inequality:** Since $\text{IoU} \leq \text{Dice}$ (harmonic mean $\leq$ arithmetic mean applied to precision and recall), we get $1 - \text{Dice} \leq 1 - \text{IoU}$, i.e., $\mathcal{L}_{\text{Dice}} \leq \mathcal{L}_{\text{IoU}}$. The upper bound follows from $\text{IoU} \geq \text{Dice}/2$. $\blacksquare$

### Step 5: RL Reward Using Differentiable IoU

In our RL formulation, the reward signal uses hard IoU (since we select discrete actions). However, the differentiable IoU connects to the Q-function gradient:

$$\nabla_\theta Q(s, a; \theta) \approx \nabla_\theta \mathbb{E}\left[\sum_{t=0}^T \gamma^t \cdot \Delta\text{IoU}_t\right]$$

This shows that Q-learning with IoU rewards implicitly optimizes a discounted version of the segmentation objective, with the discount factor $\gamma$ controlling the trade-off between immediate pixel accuracy and long-term segmentation coherence.

---
## 3. State Representation: Local Patch Extraction

Each pixel's state is the flattened intensity values of its $N \times N$ neighbourhood (zero-padded at borders).

In [ ]:
PATCH_SIZE = 5
NUM_CLASSES = 3

def extract_patch(img, i, j, patch_size=PATCH_SIZE):
    """Extract a zero-padded local patch around pixel (i, j)."""
    h, w = img.shape
    half = patch_size // 2
    patch = np.zeros((patch_size, patch_size), dtype=np.float32)
    for di in range(-half, half + 1):
        for dj in range(-half, half + 1):
            ni, nj = i + di, j + dj
            if 0 <= ni < h and 0 <= nj < w:
                patch[di + half, dj + half] = img[ni, nj]
    return patch.flatten()

def discretize_state(patch_vec, num_bins=8):
    """Discretize continuous patch into a hashable tuple for tabular Q-learning."""
    binned = np.digitize(patch_vec, bins=np.linspace(0, 1, num_bins)) 
    return tuple(binned)

sample_patch = extract_patch(image, 10, 10)
print(f'Patch shape (flattened): {sample_patch.shape}')
print(f'Discretized state length: {len(discretize_state(sample_patch))}')

## Deep Derivation: Cross-Entropy from KL Divergence and Information Theory

### Step 1: KL Divergence Definition

The Kullback-Leibler divergence from distribution $q$ to $p$ is:

$$D_{KL}(p \| q) = \sum_{k} p(k) \log \frac{p(k)}{q(k)} = \sum_k p(k) \log p(k) - \sum_k p(k) \log q(k)$$

$$= -H(p) + H(p, q)$$

where $H(p)$ is the entropy of $p$ and $H(p, q)$ is the cross-entropy.

### Step 2: Cross-Entropy as KL + Constant

Since $H(p)$ is constant w.r.t. model parameters:

$$\arg\min_\theta H(p, q_\theta) = \arg\min_\theta D_{KL}(p \| q_\theta)$$

**Proof:** $H(p, q_\theta) = D_{KL}(p \| q_\theta) + H(p)$, and $H(p)$ does not depend on $\theta$. $\blacksquare$

### Step 3: Per-Pixel Cross-Entropy Loss

For pixel $(i,j)$ with one-hot ground truth $p_{ij} = e_{y_{ij}}$ and predicted $q_{ij} = \hat{p}_{ij}$:

$$H(p_{ij}, q_{ij}) = -\sum_{k=0}^{K-1} \mathbb{1}[y_{ij} = k] \log \hat{p}_{ij,k} = -\log \hat{p}_{ij, y_{ij}}$$

The full image loss averages over all pixels:

$$\mathcal{L}_{CE} = -\frac{1}{HW}\sum_{i=1}^{H}\sum_{j=1}^{W} \log \hat{p}_{ij, y_{ij}}$$

### Step 4: Gradient of Cross-Entropy through Softmax

Let $z_{ij,k}$ be the logit (pre-softmax output) and $\hat{p}_{ij,k} = \frac{e^{z_{ij,k}}}{\sum_m e^{z_{ij,m}}}$. Then:

$$\frac{\partial \mathcal{L}_{CE}}{\partial z_{ij,k}} = \hat{p}_{ij,k} - \mathbb{1}[y_{ij} = k]$$

**Proof:**

$$\frac{\partial (-\log \hat{p}_{ij,c})}{\partial z_{ij,k}} = -\frac{1}{\hat{p}_{ij,c}} \cdot \frac{\partial \hat{p}_{ij,c}}{\partial z_{ij,k}}$$

Using the softmax Jacobian: $\frac{\partial \hat{p}_{ij,c}}{\partial z_{ij,k}} = \hat{p}_{ij,c}(\mathbb{1}[c=k] - \hat{p}_{ij,k})$

$$= -\frac{\hat{p}_{ij,c}(\mathbb{1}[c=k] - \hat{p}_{ij,k})}{\hat{p}_{ij,c}} = \hat{p}_{ij,k} - \mathbb{1}[c=k] \quad \blacksquare$$

### Step 5: Weighted Cross-Entropy for Class Imbalance in Segmentation

When background dominates ($|G_0| \gg |G_k|$ for $k > 0$), use inverse-frequency weights:

$$\mathcal{L}_{WCE} = -\frac{1}{HW}\sum_{i,j} w_{y_{ij}} \log \hat{p}_{ij, y_{ij}}, \quad w_k = \frac{N}{K \cdot n_k}$$

**Proof this equalizes per-class gradient magnitude:**

Expected gradient for class $k$: $\mathbb{E}[|\nabla_k|] \propto w_k \cdot n_k = \frac{N}{K \cdot n_k} \cdot n_k = \frac{N}{K}$

All classes contribute equally. $\blacksquare$

### Step 6: Connection to RL Reward Design

Our per-pixel reward $r = +1/-1$ is equivalent to optimizing 0-1 loss. Cross-entropy provides smoother gradients. The RL analog is **reward shaping**: instead of binary rewards, use the log-probability:

$$r_{\text{soft}}(s_{ij}, a) = \log \hat{p}_{ij, y_{ij}}$$

This is exactly the negative cross-entropy, making the RL objective equivalent to maximum likelihood estimation when $\gamma = 0$.

---
## 4. Reward Function: Per-Pixel Accuracy + IoU

$$
r_{\text{pixel}}(a, y) = \mathbb{1}[a = y] - \mathbb{1}[a \neq y]
$$

The episode-level metric is mean IoU.

In [ ]:
def pixel_reward(action, true_label):
    return 1.0 if action == true_label else -1.0

def compute_iou(pred, gt, num_classes=NUM_CLASSES):
    ious = []
    for k in range(num_classes):
        intersection = np.sum((pred == k) & (gt == k))
        union = np.sum((pred == k) | (gt == k))
        if union == 0:
            ious.append(1.0)
        else:
            ious.append(intersection / union)
    return np.mean(ious)

dummy_pred = np.zeros_like(ground_truth)
print(f'All-background mIoU: {compute_iou(dummy_pred, ground_truth):.4f}')

---
## 5. Q-Learning Agent Implementation

We use **tabular Q-learning** with an $\epsilon$-greedy exploration policy:

$$
\pi(s) = \begin{cases}
\arg\max_a Q(s,a) & \text{with probability } 1 - \epsilon \\
\text{Uniform}(\mathcal{A}) & \text{with probability } \epsilon
\end{cases}
$$

## Deep Derivation: Reward Shaping and Q-Learning Convergence Theory

### Step 1: Potential-Based Reward Shaping (Ng et al. 1999)

**Theorem:** For any MDP $\mathcal{M}$, define a shaped reward:

$$r'(s, a, s') = r(s, a, s') + \gamma \Phi(s') - \Phi(s)$$

where $\Phi: \mathcal{S} \to \mathbb{R}$ is a potential function. Then the **optimal policy is invariant** — the same policy is optimal under $r'$ as under $r$.

**Proof sketch:**

The optimal Q-function under shaped reward satisfies:

$$Q'(s, a) = Q^*(s, a) - \Phi(s) + \frac{\Phi_{\text{init}}}{1-\gamma}$$

Since $\Phi(s)$ does not depend on $a$, and the constant $\frac{\Phi_{\text{init}}}{1-\gamma}$ is action-independent:

$$\arg\max_a Q'(s, a) = \arg\max_a Q^*(s, a) \quad \blacksquare$$

### Step 2: Potential Function for Pixel Segmentation

We define $\Phi(s) = \text{mIoU}(\hat{Y}_{\text{current}})$, which measures current segmentation quality. The shaped reward becomes:

$$r'_t = r_t + \gamma \cdot \text{mIoU}_{t+1} - \text{mIoU}_t$$

This provides denser feedback: even when a single pixel reward is $-1$, if the overall segmentation improves (e.g., correcting a previously wrong pixel), the shaped reward can be positive.

### Step 3: Non-Potential Shaping Breaks Optimality

**Counterexample:** If we use $r'(s,a,s') = r(s,a,s') + F(s,a)$ where $F$ depends on $a$, the optimal policy can change.

Consider $F(s, a_1) = +100$, $F(s, a_2) = 0$. The agent will always prefer $a_1$ regardless of the true reward structure. Only potential-based shaping is provably safe. $\blacksquare$

### Step 4: Q-Learning Convergence Rate

**Theorem (Watkins & Dayan, 1992):** Tabular Q-learning converges to $Q^*$ if:

1. All state-action pairs are visited infinitely often
2. Learning rate satisfies: $\sum_t \alpha_t = \infty$ and $\sum_t \alpha_t^2 < \infty$

**Convergence rate** with constant learning rate $\alpha$:

$$\|Q_T - Q^*\|_\infty \leq (1 - \alpha(1-\gamma))^T \|Q_0 - Q^*\|_\infty + \frac{\alpha \gamma V_{\max}}{1 - \gamma}$$

### Step 5: Convergence Rate for Pixel Classification

In our pixel classification MDP:
- State space size: $|\mathcal{S}| = B^{N^2 C}$ (exponential in patch size)
- Action space size: $|\mathcal{A}| = K$ (number of classes)

For $\epsilon$-optimal Q-values, we need:

$$T = O\left(\frac{|\mathcal{S}||\mathcal{A}|}{\alpha(1-\gamma)^2} \log\frac{|\mathcal{S}||\mathcal{A}|}{\epsilon}\right)$$

With $B=8$, $N=5$, $C=1$, $K=3$: $|\mathcal{S}| = 8^{25} \approx 3 \times 10^{22}$. This astronomical state space is precisely why we use **discretization and function approximation** in practice.

### Step 6: State Space Reduction via Tile Coding

**Tile coding** represents states using $M$ overlapping tilings, each with $T$ tiles per dimension:

$$\phi(s) = [\mathbb{1}[s \in \text{tile}_{m,i}]]_{m=1,\ldots,M;\; i=1,\ldots,T^d}$$

**Effective resolution:** $\frac{1}{M \cdot T}$ per dimension (finer than single tiling).

**Approximation guarantee:** With $M$ tilings, the function approximation error satisfies:

$$\|Q - Q_{\text{approx}}\|_\infty \leq \frac{L}{M \cdot T}$$

where $L$ is the Lipschitz constant of $Q$. $\blacksquare$

### Step 7: Radial Basis Function (RBF) State Representation

An alternative to discretization uses Gaussian RBFs centered at $\{c_i\}$:

$$\phi_i(s) = \exp\left(-\frac{\|s - c_i\|^2}{2\sigma^2}\right)$$

$$Q(s, a) = \sum_i w_{i,a} \cdot \phi_i(s)$$

**Advantage:** Smooth interpolation between states. **Disadvantage:** Requires choosing center locations and bandwidth $\sigma$.

For pixel segmentation, RBF centers can be placed at cluster centroids of observed patch vectors, providing natural state abstraction.

In [ ]:
class PixelQLearningAgent:
    def __init__(self, num_actions=NUM_CLASSES, alpha=0.1, gamma=0.9, epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.05):
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.q_table = defaultdict(lambda: np.zeros(num_actions))

    def select_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.num_actions)
        return int(np.argmax(self.q_table[state]))

    def update(self, state, action, reward, next_state):
        best_next = np.max(self.q_table[next_state])
        td_error = reward + self.gamma * best_next - self.q_table[state][action]
        self.q_table[state][action] += self.alpha * td_error

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

---
## 6. Training Loop

Each episode scans every pixel in the image. The agent extracts the local patch, selects a class, receives a reward, and updates its Q-table.

In [ ]:
def train_pixel_agent(image, ground_truth, agent, num_episodes=80):
    h, w = image.shape
    history = {'episode': [], 'miou': [], 'reward': [], 'epsilon': []}
    snapshots = {}

    for ep in range(num_episodes):
        pred_map = np.zeros((h, w), dtype=np.int64)
        total_reward = 0.0
        pixels = [(i, j) for i in range(h) for j in range(w)]
        np.random.shuffle(pixels)

        for idx, (i, j) in enumerate(pixels):
            patch = extract_patch(image, i, j)
            state = discretize_state(patch)
            action = agent.select_action(state)
            reward = pixel_reward(action, ground_truth[i, j])

            if idx + 1 < len(pixels):
                ni, nj = pixels[idx + 1]
                next_patch = extract_patch(image, ni, nj)
                next_state = discretize_state(next_patch)
            else:
                next_state = state

            agent.update(state, action, reward, next_state)
            pred_map[i, j] = action
            total_reward += reward

        agent.decay_epsilon()
        miou = compute_iou(pred_map, ground_truth)
        history['episode'].append(ep)
        history['miou'].append(miou)
        history['reward'].append(total_reward)
        history['epsilon'].append(agent.epsilon)

        if ep in [0, 9, 19, 39, 59, 79] or ep == num_episodes - 1:
            snapshots[ep] = pred_map.copy()

        if ep % 10 == 0:
            print(f'Episode {ep:3d} | mIoU: {miou:.4f} | Reward: {total_reward:8.1f} | ε: {agent.epsilon:.4f}')

    return history, snapshots

agent = PixelQLearningAgent(num_actions=NUM_CLASSES, alpha=0.15, gamma=0.5, epsilon=1.0, epsilon_decay=0.97, epsilon_min=0.05)
history, snapshots = train_pixel_agent(image, ground_truth, agent, num_episodes=80)
print('\nTraining complete.')

---
## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['episode'], history['miou'], color='teal', linewidth=2)
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Mean IoU'); axes[0].set_title('Segmentation Quality (mIoU)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['episode'], history['reward'], color='coral', linewidth=2)
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Total Reward'); axes[1].set_title('Cumulative Reward per Episode')
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['episode'], history['epsilon'], color='purple', linewidth=2)
axes[2].set_xlabel('Episode'); axes[2].set_ylabel('Epsilon'); axes[2].set_title('Exploration Rate Decay')
axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 8. Segmentation Progress Over Episodes

## Deep Derivation: Pixel-Level Accuracy Metrics and Confusion Matrix Analysis

### Step 1: Confusion Matrix for Segmentation

For $K$ classes, the confusion matrix $\mathbf{C} \in \mathbb{R}^{K \times K}$ has entries:

$$C_{ij} = |\{(r,c) : y_{rc} = i \text{ and } \hat{y}_{rc} = j\}|$$

All segmentation metrics can be derived from $\mathbf{C}$:

$$\text{TP}_k = C_{kk}, \quad \text{FP}_k = \sum_{i \neq k} C_{ik}, \quad \text{FN}_k = \sum_{j \neq k} C_{kj}$$

### Step 2: Per-Class Precision and Recall

$$\text{Precision}_k = \frac{C_{kk}}{\sum_i C_{ik}} = \frac{\text{TP}_k}{\text{TP}_k + \text{FP}_k}$$

$$\text{Recall}_k = \frac{C_{kk}}{\sum_j C_{kj}} = \frac{\text{TP}_k}{\text{TP}_k + \text{FN}_k}$$

### Step 3: Relationship Between IoU, Precision, and Recall

$$\text{IoU}_k = \frac{\text{TP}_k}{\text{TP}_k + \text{FP}_k + \text{FN}_k}$$

**Proof IoU ≤ min(Precision, Recall):**

$$\text{IoU}_k = \frac{\text{TP}_k}{\text{TP}_k + \text{FP}_k + \text{FN}_k} \leq \frac{\text{TP}_k}{\text{TP}_k + \text{FP}_k} = \text{Precision}_k$$

Similarly $\text{IoU}_k \leq \text{Recall}_k$. Therefore $\text{IoU}_k \leq \min(\text{Precision}_k, \text{Recall}_k)$. $\blacksquare$

### Step 4: Frequency-Weighted IoU (FWIoU)

$$\text{FWIoU} = \frac{1}{\sum_k t_k} \sum_{k=0}^{K-1} t_k \cdot \text{IoU}_k$$

where $t_k = \sum_j C_{kj}$ is the total number of pixels of class $k$.

**Why FWIoU vs mIoU?** FWIoU weights classes by their frequency, giving more importance to dominant classes. mIoU treats all classes equally — better for detecting rare objects.

### Step 5: Cohen's Kappa for Segmentation Agreement

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where $p_o = \frac{\sum_k C_{kk}}{N}$ (observed agreement) and $p_e = \frac{\sum_k (\sum_i C_{ik})(\sum_j C_{kj})}{N^2}$ (chance agreement).

**Proof $\kappa = 1$ iff perfect agreement:** If $C_{ij} = 0$ for $i \neq j$, then $p_o = 1$ and $\kappa = \frac{1 - p_e}{1 - p_e} = 1$. $\blacksquare$

### Step 6: Error Map Interpretation Through Bayesian Analysis

The error map shows misclassified pixels. The posterior probability of the true class given the patch:

$$P(y_{ij} = k \mid s_{ij}) = \frac{P(s_{ij} \mid y_{ij} = k) P(y_{ij} = k)}{P(s_{ij})}$$

Errors concentrate where class-conditional distributions $P(s \mid y=k_1)$ and $P(s \mid y=k_2)$ overlap — typically at object boundaries. The Q-learning agent implicitly learns these posterior probabilities through the Q-values:

$$\pi^*(s) = \arg\max_k Q^*(s, k) \approx \arg\max_k P(y = k \mid s) \quad \text{(for } \gamma \to 0\text{)}$$

In [ ]:
snap_keys = sorted(snapshots.keys())
n_snaps = len(snap_keys)
fig, axes = plt.subplots(1, n_snaps + 1, figsize=(3.5 * (n_snaps + 1), 3.5))

axes[0].imshow(ground_truth, cmap=cmap_gt, vmin=0, vmax=2)
axes[0].set_title('Ground Truth'); axes[0].axis('off')

for idx, ep in enumerate(snap_keys):
    axes[idx + 1].imshow(snapshots[ep], cmap=cmap_gt, vmin=0, vmax=2)
    iou_val = compute_iou(snapshots[ep], ground_truth)
    axes[idx + 1].set_title(f'Ep {ep} (mIoU={iou_val:.2f})')
    axes[idx + 1].axis('off')

plt.suptitle('Segmentation Progress Over Training', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 9. Final Comparison with Ground Truth

In [ ]:
def evaluate_agent(image, ground_truth, agent):
    h, w = image.shape
    pred_map = np.zeros((h, w), dtype=np.int64)
    for i in range(h):
        for j in range(w):
            patch = extract_patch(image, i, j)
            state = discretize_state(patch)
            pred_map[i, j] = int(np.argmax(agent.q_table[state]))
    return pred_map

final_pred = evaluate_agent(image, ground_truth, agent)
final_iou = compute_iou(final_pred, ground_truth)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Input Image'); axes[0].axis('off')
axes[1].imshow(ground_truth, cmap=cmap_gt, vmin=0, vmax=2); axes[1].set_title('Ground Truth'); axes[1].axis('off')
axes[2].imshow(final_pred, cmap=cmap_gt, vmin=0, vmax=2); axes[2].set_title(f'Agent Prediction (mIoU={final_iou:.3f})'); axes[2].axis('off')
plt.suptitle('Final Segmentation Comparison', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

for k, name in enumerate(['Background', 'Circle', 'Square']):
    inter = np.sum((final_pred == k) & (ground_truth == k))
    union = np.sum((final_pred == k) | (ground_truth == k))
    iou_k = inter / union if union > 0 else 1.0
    print(f'  Class {k} ({name:>10s}): IoU = {iou_k:.4f}')
print(f'  Mean IoU: {final_iou:.4f}')

---
## 10. Pixel-Level Error Map

In [ ]:
error_map = (final_pred != ground_truth).astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(error_map, cmap='Reds', vmin=0, vmax=1)
axes[0].set_title(f'Error Map ({np.sum(error_map > 0)} misclassified pixels)')
axes[0].axis('off')

pixel_acc = np.mean(final_pred == ground_truth)
class_names = ['Background', 'Circle', 'Square']
class_acc = [np.mean(final_pred[ground_truth == k] == k) if np.sum(ground_truth == k) > 0 else 0 for k in range(NUM_CLASSES)]
bars = axes[1].bar(class_names, class_acc, color=['gray', 'dodgerblue', 'orange'], edgecolor='black')
axes[1].set_ylabel('Accuracy'); axes[1].set_title(f'Per-Class Accuracy (Overall: {pixel_acc:.3f})')
axes[1].set_ylim(0, 1.1)
for bar, acc in zip(bars, class_acc):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{acc:.2f}', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

---
## Summary

In this notebook we:

1. **Formulated pixel classification as an MDP** — each pixel is a decision point with a local-patch state, class-label actions, and an IoU-based reward signal.
2. **Built a tabular Q-learning agent** that learns which class to assign to each pixel by examining its $5 \times 5$ neighbourhood.
3. **Trained the agent** on a synthetic image with three classes (background, circle, square) and observed monotonic improvement in mIoU.
4. **Visualized the segmentation trajectory** — from random labelling to accurate region identification.
5. **Analysed per-class IoU and pixel accuracy**, confirming the agent successfully distinguishes all three regions.

### Key Takeaways

| Concept | Detail |
|---|---|
| State | Flattened $N \times N$ local patch |
| Action | Class label $\{0, \ldots, K-1\}$ |
| Reward | $+1$ correct, $-1$ incorrect |
| Algorithm | Tabular Q-learning with $\epsilon$-greedy |
| Metric | Mean Intersection over Union (mIoU) |

**Next →** Notebook 02 explores a **Region Growing Agent** that iteratively expands segmentation regions.